# nano Beatrix — the scale test's members: real arms on Alhena and Castor

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_scale_members.ipynb)

**What trains**: on each of the four grid cores (Alhena and Castor, two seeds; trained by `nano_scale_cores.ipynb`), independently trained arms, each owning one address of the window, then merged under the product gate with no co-training and no learned router:

| stage | members | where they write | budget |
|---|---|---|---|
| last site | 48 real arms (the arm library's RelayPatchwork, 16 slots x 8 through an aleph read of 16 anchors, born inert) + 48 toy members (the replication arm) | after the last block, before the final norm | 16,000 steps, each member 3 of its questions a step |
| mid site | the first 16 owners as real arms | after block 3 of 6 (blocks 4-6 mix positions after the write) | 32,000 batch steps, one row a member |

Each member answers a question on its own record in its own letter code (a fixed derangement of a-z), trained ALONE with its own Adam on the frozen core. The read merges 16 / 32 / 48 of them under the gate w_i = sinh(u_i) / (1 + sum_j (cosh(u_j) - 1)), u_i = U k_1 k_2 (the request's two digits against the member's), with the request from the layout, the core's own written index, a corrupted index, the core's address head (plain and floored), the oracle and the ungated control, and grades three marks per cell: PROTECTED (the merged member keeps its solo answers), QUIET (the unowned questions keep the bare core's answers), EXACT (both, to .01 and a KL of .001).

**The cells**: C1 pins the trainer and the arm library and holds the decision block; C2 builds the data, fetches the cores, checks the compiled writes against eager at step 1 on this card, prices 200 steps of each trainer and probes the push path; C3 runs the cores in order, every finished core's members shipped to [AbstractPhil/alephllm-nano-beatrix-training](https://huggingface.co/AbstractPhil/alephllm-nano-beatrix-training) under `bed/scale_members/` (the stacks as safetensors, the records as JSON); C4 renders the table.

**Before C1**: the notebook's `HF_TOKEN` secret (write access to that repo) with notebook access on. A file named `STOP` in `/content/out` ends the session between cores; a restarted session resumes (each trainer checkpoints every few thousand steps). The code and its tests live in [AbstractEyes/geolip-nano-beatrix-trainer](https://github.com/AbstractEyes/geolip-nano-beatrix-trainer) (`geolip/nano_beatrix/rotary_scale.py`).

In [ ]:
# C1 — pinned installs + THE DECISION BLOCK. RESTART RUNTIME if the version line below changes.
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.71" "amoe-lora @ git+https://github.com/AbstractEyes/amoe-lora@v0.2.12"
import torch, amoe
import geolip.nano_beatrix as N
print('geolip.nano_beatrix', N.__version__, '| amoe', amoe.__version__, '| torch', torch.__version__, '| card',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 71), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart, then rerun from C1')
assert amoe.__version__ == '0.2.12', 'the arm module of record is amoe-lora 0.2.12'
assert torch.cuda.is_available(), 'no card: Runtime > Change runtime type > GPU'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN            # the shipper reads it from the environment; the value is never printed

# ============ THE DECISION BLOCK — every setting set from the record, with its source beside it ============
OUT = '/content/out'
REPO = 'AbstractPhil/alephllm-nano-beatrix-training'  # the cores come from here; every finished core's members ship here (bed/scale_members/)
CORPUS_BYTES = 160e6                                   # the bed corpus (the members' training rows draw their filler from it)
TAIL = '_ctx4096_frdyn_cap_mode_sz32-128-256-512_r4_n30_o0_askany_gag50_ahdigit1_scale'
CORE_STEPS = {1: 24000, 2: 24000}                     # the registered retrains (the plan's section 8): both seeds missed the written index at
                                                       # 12,000 steps, so the members train on the 24,000-step cores
CORES = [(f'{arm}_s{seed}_L{CORE_STEPS[seed]}{TAIL}', seed)  # the claim's core first in each seed (Alhena), then the core without the rotary
         for seed in (1, 2) for arm in ('alhena-both', 'castor-both')]
LAST_STEPS = 16000                                     # the S-line's member budget after its amendment M (16,000 steps; 4,000 left owners on a plateau)
MID_STEPS = 32000                                      # batch steps at the mid site: one row a member, half of the 16 a step = 16,000 own steps a member
READ_BATCHES, CALIB_BATCHES = 64, 32                   # the cores' own read and calibration rows (8 rows a batch; ~190 questions an address)
PRICE_STEPS = 200                                      # C2's price lines (each trainer on throwaway stacks)
print('decision block:', len(CORES), 'cores |', LAST_STEPS, 'last-site steps |', MID_STEPS, 'mid-site batch steps | ships to',
      REPO if HF_TOKEN else 'nothing (tokenless)')

In [ ]:
# C2 (P) — PREFLIGHT: the data, the cores, compiled = eager at step 1 on this card, the price lines, the push probe. No member trains.
import json, time
from huggingface_hub import snapshot_download
from geolip.nano_beatrix import bed_corpus, rotary_scale as rs
BED_DIR = f'{OUT}/bed'
if not os.path.exists(f'{BED_DIR}/bed_corpus.bin'):
    bed_corpus.build(BED_DIR, CORPUS_BYTES)
HELD = f'{BED_DIR}/heldout_fineweb_edu_head2048.jsonl'
print('corpus', os.path.getsize(f'{BED_DIR}/bed_corpus.bin') / 1e6, 'MB; held-out head', os.path.exists(HELD))
CORE_DIRS = {}
for tag, seed in CORES:
    snapshot_download(REPO, allow_patterns=[f'bed/weights/{tag}/*'], local_dir=f'{OUT}/hub', token=HF_TOKEN or False)
    CORE_DIRS[tag] = f'{OUT}/hub/bed/weights/{tag}'
    assert all(os.path.exists(f'{CORE_DIRS[tag]}/{f}') for f in ('config.json', 'model.safetensors', 'scale.json', 'addr_head.safetensors')), tag
print('cores:', list(CORE_DIRS))
owned, never = rs.owned_sets()
packed = rs.load_packed(BED_DIR)
core = rs.Core(CORE_DIRS[CORES[0][0]], 'cuda')
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
cc = rs.compile_check(core, packed, owned, never)
print('compiled = eager at step 1 (largest difference over the largest value):', cc)
assert all(v['write'] < 1e-4 and v['grad'] < 1e-3 for v in cc.values()), 'the compiled writes are not the eager writes: no member trains'
pr = rs.price(core, packed, owned, never, steps=PRICE_STEPS)
per_core = (pr['last_ms_per_step'] * LAST_STEPS + pr['mid_ms_per_step'] * MID_STEPS) / 60000
print(f"this card: last site {pr['last_ms_per_step']:.1f} ms a step, mid site {pr['mid_ms_per_step']:.1f} ms a step -> a core's members about "
      f"{per_core:.0f} min + the reads; the {len(CORES)} cores about {per_core * len(CORES) / 60:.1f} h + reads (the first steps carry the compile)")
del core
torch.cuda.empty_cache()
PUSH_OK = False
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    path = 'bed/preflight/push_probe_scale_members.json'
    body = json.dumps({'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'card': torch.cuda.get_device_name(0),
                       'compile_check': cc, 'price': pr, 'cores': [t for t, _ in CORES]}).encode()
    api.upload_file(path_or_fileobj=body, path_in_repo=path, repo_id=REPO, commit_message='nano scale members: push probe')
    PUSH_OK = path in set(api.list_repo_files(REPO))
    print('push probe', 'READ BACK OK' if PUSH_OK else 'NOT VISIBLE after upload', path)
else:
    print('TOKENLESS: no probe written; C3 runs and ships nothing')

In [ ]:
# C3 (T) — THE SESSION: the cores in order, each core's last site (train, read, text bits) then its mid site (train, read), resume-first;
# a STOP file in OUT starts no new core; every finished core's folder ships in the background (the card never waits on the Hub).
shipper = rs.Shipper(REPO) if (HF_TOKEN and PUSH_OK) else None
done = []
for n, (tag, seed) in enumerate(CORES, 1):
    if os.path.exists(f'{OUT}/STOP'):
        print('STOP present: no new core starts')
        break
    print(f'=================== {n}/{len(CORES)} {tag} ===================')
    t0 = time.time()
    res = rs.run_core(CORE_DIRS[tag], f'{OUT}/members', seed, BED_DIR, HELD, last_steps=LAST_STEPS, mid_steps=MID_STEPS,
                      read_batches=READ_BATCHES, calib_batches=CALIB_BATCHES, shipper=shipper)
    done.append(res)
    torch.cuda.empty_cache()
    print(f'{tag}: {(time.time() - t0) / 60:.1f} min; {len(CORES) - n} cores left')
if shipper:
    print('ships drained' if shipper.drain() else 'ships still retrying in the background')
print('session:', [r['name'] for r in done])

In [ ]:
# C4 — THE TABLE: per core, the real arms' and the toy members' solo against the bare core, the claim's cells (INDEX, HEADP, HEADPQ999 at
# the nine rest cells: how many keep all three marks), the controls, the largest leak, the mid site, the text bits. Run any time.
import glob
from IPython.display import Markdown, display
rows = ['| core | stack | solo / bare owned | INDEX 9 cells | HEADP | HEADPQ999 | LAYOUT | INDEXC m48 Q | G0 m48 P/Q | largest leak / own | mid solo | mid INDEX U5 P/Q/E | G0 text bits |',
        '|---|---|---|---|---|---|---|---|---|---|---|---|---|']
for d in sorted(glob.glob(f'{OUT}/members/*_members_*')):
    if not os.path.exists(f'{d}/last.json'):
        continue
    L = json.load(open(f'{d}/last.json', encoding='utf-8'))
    M = json.load(open(f'{d}/mid.json', encoding='utf-8')) if os.path.exists(f'{d}/mid.json') else None
    rd = L['read']
    for st, r in rd['stacks'].items():
        c = r['cells']
        nine = lambda s: sum(all(c[f'{s}_U{u}_REST_m{m}'][k] for k in ('PROTECTED', 'QUIET', 'EXACT')) for u in (5, 8, 10) for m in (16, 32, 48))
        leak = max(c[f'{s}_U{u}_REST_m{m}']['leak_over_own_write'] for s in ('INDEX', 'HEADP', 'HEADPQ999') for u in (5, 8, 10) for m in (16, 32, 48))
        mid = '-'
        if M and st == 'arms':
            mc = M['read']['cells'].get('INDEX_U5_REST_m16', {})
            mid = f"{M['read']['solo']['ONE']:.3f} | {mc.get('PROTECTED')}/{mc.get('QUIET')}/{mc.get('EXACT')}"
        else:
            mid = '- | -'
        g0 = L['text_bits'].get(f'G0_{st}_minus_bare')
        vac = rd['calibration'].get('tau_999_saturated') or rd['floor']['owners_floored_to_0'] > 0.99    # the floor silences the owners
        q999 = f"vacuous ({rd['floor']['owners_floored_to_0']:.2f} floored)" if vac else f"{nine('HEADPQ999')}/9"
        rows.append(f"| {L['core'][:24]} | {st} | {r['solo']['ONE']['48']:.3f} / {rd['bare']['owned']:.3f} | {nine('INDEX')}/9 | {nine('HEADP')}/9 | "
                    f"{q999} | {nine('LAYOUT')}/9 | {c['INDEXC_U5_REST_m48']['QUIET']} | {c['G0_m48']['PROTECTED']}/{c['G0_m48']['QUIET']} | "
                    f"{leak:.1e} | {mid} | {g0:+.3f} |")
display(Markdown('\n'.join(rows)))